# PredictiX — Breakdown Cost Estimation Model v4.0
**Target:** `maintenance_cost_lkr_next_30d` (cost IF the asset has maintenance in next 30 days)  
**Dataset:** `srilanka_single_warehouse_vehicle_maintenance_dataset_v11_realistic.csv`  
**Scope:** Predict the **breakdown/corrective cost** an asset will incur based on its current  
health signals — oil life, tire health, battery voltage, days overdue, lifetime breakdown history.

This is different from v3 (which predicted the *last* service cost).  
v4 predicts *forward-looking* cost given current health degradation signals.

---
### Why this model?
The asset detail page shows:
- **SHAP risk factors** from the *failure model* (oil life, battery voltage, days since service)
- **Est. Maintenance Cost** — this should reflect those *same risk signals*
- v3 predicted cost of the *most recent past service* — not the *next expected cost*

v4 trains on `maintenance_cost_lkr_next_30d` for rows where `maintenance_required_next_30d == 1`,  
so the model learns: **given current health state → what will it cost when it next breaks/services?**


In [ ]:
!pip install catboost lightgbm xgboost shap -q

import numpy as np, pandas as pd, time, json, warnings, joblib
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import lightgbm as lgb, shap

plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.3,
                     'axes.facecolor':'white','figure.facecolor':'white'})

SEEDS  = [42, 1337, 2026]
TARGET = 'maintenance_cost_lkr_next_30d'
MODEL_VERSION = 'breakdown-cost-v4.0'
print('Libraries loaded.')

In [ ]:
# ── Load & filter ─────────────────────────────────────────────────────────────
DATA_PATH = 'srilanka_single_warehouse_vehicle_maintenance_dataset_v11_realistic.csv'
df_raw = pd.read_csv(DATA_PATH)
print(f'Raw shape: {df_raw.shape}')

# Keep only rows where maintenance IS required in next 30 days
# These are the rows where the target is meaningful
df = df_raw[df_raw['maintenance_required_next_30d'] == 1].copy()
df = df[df[TARGET] > 0].copy()
print(f'After filter (maintenance_required=1, cost>0): {df.shape}')
print(f'Target stats: mean={df[TARGET].mean():,.0f}  median={df[TARGET].median():,.0f}  '
      f'p25={df[TARGET].quantile(0.25):,.0f}  p75={df[TARGET].quantile(0.75):,.0f}')
print(f'Skew: {df[TARGET].skew():.2f}')

# Cost distribution by vehicle type
print('\nCost by vehicle_type:')
print(df.groupby('vehicle_type')[TARGET].agg(['count','mean','median']).round(0).sort_values('mean',ascending=False))


In [ ]:
# ── Time-based split ──────────────────────────────────────────────────────────
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])
df = df.sort_values('snapshot_date').reset_index(drop=True)
n = len(df); a, b = int(n*0.70), int(n*0.85)
train_df = df.iloc[:a].copy()
val_df   = df.iloc[a:b].copy()
test_df  = df.iloc[b:].copy()
print(f'Train: {len(train_df):,}  ({train_df.snapshot_date.min().date()} → {train_df.snapshot_date.max().date()})')
print(f'Val  : {len(val_df):,}  ({val_df.snapshot_date.min().date()} → {val_df.snapshot_date.max().date()})')
print(f'Test : {len(test_df):,}  ({test_df.snapshot_date.min().date()} → {test_df.snapshot_date.max().date()})')

# Winsorise at 99.5th percentile of TRAIN only
cap = float(train_df[TARGET].quantile(0.995))
n0 = len(train_df)
train_df = train_df[train_df[TARGET] <= cap].copy()
print(f'\nWinsorise cap: LKR {cap:,.0f}  dropped {n0-len(train_df)} rows')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
VEHICLE_SIZE_MAP = {
    'Mini_Truck_1T':1,'Delivery_Van_1.5T':2,'Light_Truck_3.5T':3,
    'Medium_Truck_7T':4,'Heavy_Truck_16T':5,'Forklift_2.5T':3,'Forklift_3.0T':4
}
PART_TOKENS = ['oil_filter','engine_oil','brake_pads','discs','fluid','tires','valves',
               'hydraulic_pump','seals','battery','terminals','radiator','coolant',
               'hoses','injectors','gaskets','engine','clutch','mounts']

def engineer(df_in):
    d = df_in.copy()
    age = d['vehicle_age_years'].fillna(0)
    d['vehicle_size_ord']   = d['vehicle_type'].map(VEHICLE_SIZE_MAP).fillna(3).astype(int)
    d['age_band']           = pd.cut(age, bins=[0,3,6,10,100], labels=['new','mid','old','aged']).astype(str)
    d['service_vehicle']    = d['next_service_type'].astype(str) + '__' + d['vehicle_type'].astype(str)
    d['payload_age']        = d['payload_capacity_kg'].fillna(0) * age
    d['utilisation_stress'] = d['payload_utilization_pct'].fillna(0) * d['downtime_hours_last_90d'].fillna(0)
    d['health_index']       = (d['oil_life_pct'].fillna(50) + d['brake_health_pct'].fillna(50) +
                               d['tire_health_pct'].fillna(50) + d['battery_health_pct'].fillna(50)) / 4
    d['fault_stress']       = d['active_fault_code_count'].fillna(0) + d['sensor_fault_flag'].fillna(0)*3
    # Health degradation signals (key for breakdown cost)
    d['health_deficit']     = 100 - d['health_index']
    d['overdue_days']       = (d['days_since_last_service'] - 60).clip(lower=0)
    d['breakdown_history']  = d['lifetime_breakdown_count'].fillna(0)
    d['service_intensity']  = d['lifetime_service_count'].fillna(0) / (age + 1)
    for tok in PART_TOKENS:
        d[f'part_{tok}'] = d['parts_replaced_last_service'].astype(str).str.contains(tok, na=False).astype(int)
    return d

for split in [train_df, val_df, test_df]:
    split_e = engineer(split)
for nm, s in [('train',train_df),('val',val_df),('test',test_df)]:
    globals()[f'{nm}_df'] = engineer(s)
print(f'After FE: {train_df.shape}')


In [ ]:
# ── Feature selection ─────────────────────────────────────────────────────────
DROP = [
    'vehicle_id','snapshot_date','warehouse_id','warehouse_name','warehouse_city',
    'warehouse_type','climate_zone','is_home_warehouse_service',
    # leakage — these are FUTURE values
    'maintenance_cost_last_service_lkr',
    'maintenance_required_next_30d',
    'maintenance_cost_lkr_next_30d',
    'next_service_type',          # future
    'days_until_next_maintenance',# future
    'predicted_next_maintenance_date', # future
    'spare_parts_delay_days',     # future
    'parts_replaced_last_service',# past service — not available for future prediction
    'age_band',
    TARGET,
]
DROP = [c for c in DROP if c in train_df.columns]
feature_cols = [c for c in train_df.columns if c not in DROP]

X_train = train_df[feature_cols].copy(); y_train = train_df[TARGET].values.astype(float)
X_val   = val_df[feature_cols].copy();   y_val   = val_df[TARGET].values.astype(float)
X_test  = test_df[feature_cols].copy();  y_test  = test_df[TARGET].values.astype(float)

# log1p target
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

numeric_cols     = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f'Features: {len(feature_cols)} (numeric={len(numeric_cols)}, cat={len(categorical_cols)})')
print(f'Categorical: {categorical_cols}')

corrs = X_train[numeric_cols].corrwith(pd.Series(y_train_log, index=X_train.index)).abs().sort_values(ascending=False)
print(f'\nTop 10 |corr| with log1p(target):')
print(corrs.head(10).round(3).to_string())
print(f'Max |corr|: {corrs.max():.3f}  {"WARN leakage" if corrs.max() > 0.95 else "OK"}')

In [ ]:
# ── Preprocessors ─────────────────────────────────────────────────────────────
def prep_gbm(X, dtypes=None):
    Xg = X.copy()
    for c in categorical_cols:
        s = Xg[c].astype(str).fillna('__missing__')
        Xg[c] = s.astype(dtypes[c]) if dtypes else s.astype('category')
    for c in numeric_cols:
        med = X_train[c].median()
        Xg[c] = Xg[c].fillna(med if pd.notna(med) else 0.0)
    return Xg

def prep_cb(X):
    Xc = X.copy()
    for c in categorical_cols:
        Xc[c] = Xc[c].astype(str).fillna('__missing__')
    for c in numeric_cols:
        med = X_train[c].median()
        Xc[c] = Xc[c].fillna(med if pd.notna(med) else 0.0)
    return Xc

X_train_g = prep_gbm(X_train)
gbm_dtypes_safe = {c: list(X_train_g[c].cat.categories) for c in categorical_cols}
X_val_g   = prep_gbm(X_val,  {c: pd.CategoricalDtype(cats, ordered=False) for c,cats in gbm_dtypes_safe.items()})
X_test_g  = prep_gbm(X_test, {c: pd.CategoricalDtype(cats, ordered=False) for c,cats in gbm_dtypes_safe.items()})
X_train_cb, X_val_cb, X_test_cb = prep_cb(X_train), prep_cb(X_val), prep_cb(X_test)
print('Preprocessing done.')

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
def reg_metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mede = np.median(np.abs(y_true - y_pred))
    m    = y_true > 100
    mape = np.mean(np.abs((y_true[m]-y_pred[m])/y_true[m]))*100 if m.sum() else np.nan
    if label:
        print(f'  {label:12s} | MAE={mae:>10,.0f} | RMSE={rmse:>10,.0f} | '
              f'R²={r2:.4f} | MedAE={mede:>8,.0f} | MAPE={mape:5.1f}%')
    return {'MAE':mae,'RMSE':rmse,'R²':r2,'MedAE':mede,'MAPE%':mape}

In [ ]:
# ── Bake-off (log1p target) ────────────────────────────────────────────────────
results = {}

print('='*72+'\nLightGBM\n'+'='*72)
runs=[]
for seed in SEEDS:
    t0=time.time()
    m = LGBMRegressor(n_estimators=2000,max_depth=9,num_leaves=127,learning_rate=0.02,
                      subsample=0.8,colsample_bytree=0.75,reg_lambda=5.0,min_child_samples=20,
                      objective='regression',random_state=seed,n_jobs=-1,verbose=-1)
    m.fit(X_train_g,y_train_log,eval_set=[(X_val_g,y_val_log)],eval_metric='rmse',
          categorical_feature=categorical_cols,
          callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(0)])
    vp=np.expm1(np.clip(m.predict(X_val_g),0,None)); tp=np.expm1(np.clip(m.predict(X_test_g),0,None))
    vm=mean_absolute_error(y_val,vp)
    runs.append({'seed':seed,'model':m,'val_pred':vp,'test_pred':tp,'val_mae':vm,'fit_time':time.time()-t0})
    print(f'  Seed {seed}: val_MAE={vm:>10,.0f} | {runs[-1]["fit_time"]:.0f}s | best={m.best_iteration_}')
results['LightGBM']=min(runs,key=lambda r:r['val_mae'])

print('='*72+'\nCatBoost\n'+'='*72)
runs=[]
for seed in SEEDS:
    t0=time.time()
    m=CatBoostRegressor(iterations=2000,depth=9,learning_rate=0.02,loss_function='RMSE',
                        eval_metric='RMSE',random_seed=seed,verbose=0,early_stopping_rounds=100,
                        l2_leaf_reg=5.0,min_data_in_leaf=20)
    m.fit(X_train_cb,y_train_log,cat_features=categorical_cols,eval_set=(X_val_cb,y_val_log),use_best_model=True)
    vp=np.expm1(np.clip(m.predict(X_val_cb),0,None)); tp=np.expm1(np.clip(m.predict(X_test_cb),0,None))
    vm=mean_absolute_error(y_val,vp)
    runs.append({'seed':seed,'model':m,'val_pred':vp,'test_pred':tp,'val_mae':vm,'fit_time':time.time()-t0})
    print(f'  Seed {seed}: val_MAE={vm:>10,.0f} | {runs[-1]["fit_time"]:.0f}s | best={m.best_iteration_}')
results['CatBoost']=min(runs,key=lambda r:r['val_mae'])

print('='*72+'\nXGBoost\n'+'='*72)
runs=[]
for seed in SEEDS:
    t0=time.time()
    m=XGBRegressor(n_estimators=2000,max_depth=9,learning_rate=0.02,subsample=0.8,
                   colsample_bytree=0.75,reg_lambda=5.0,min_child_weight=20,
                   tree_method='hist',enable_categorical=True,eval_metric='rmse',
                   early_stopping_rounds=100,random_state=seed,n_jobs=-1)
    m.fit(X_train_g,y_train_log,eval_set=[(X_val_g,y_val_log)],verbose=False)
    vp=np.expm1(np.clip(m.predict(X_val_g),0,None)); tp=np.expm1(np.clip(m.predict(X_test_g),0,None))
    vm=mean_absolute_error(y_val,vp)
    runs.append({'seed':seed,'model':m,'val_pred':vp,'test_pred':tp,'val_mae':vm,'fit_time':time.time()-t0})
    print(f'  Seed {seed}: val_MAE={vm:>10,.0f} | {runs[-1]["fit_time"]:.0f}s | best={m.best_iteration}')
results['XGBoost']=min(runs,key=lambda r:r['val_mae'])

In [ ]:
# ── Test comparison ────────────────────────────────────────────────────────────
print('\n'+'─'*72+'\nTEST-SET METRICS\n'+'─'*72)
test_metrics={n:reg_metrics(y_test,results[n]['test_pred'],n) for n in ['LightGBM','CatBoost','XGBoost']}
PREDICTOR=min(test_metrics.items(),key=lambda kv:kv[1]['MAE'])[0]
print(f'\n→ WINNER: {PREDICTOR}  (test MAE={test_metrics[PREDICTOR]["MAE"]:,.0f}  R²={test_metrics[PREDICTOR]["R²"]:.4f})')

In [ ]:
# ── Quantile models ────────────────────────────────────────────────────────────
def train_q(alpha):
    m=LGBMRegressor(n_estimators=2000,max_depth=9,num_leaves=127,learning_rate=0.02,
                    subsample=0.8,colsample_bytree=0.75,reg_lambda=5.0,min_child_samples=20,
                    objective='quantile',alpha=alpha,random_state=42,n_jobs=-1,verbose=-1)
    m.fit(X_train_g,y_train_log,eval_set=[(X_val_g,y_val_log)],eval_metric='quantile',
          categorical_feature=categorical_cols,
          callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(0)])
    return m

print('Training quantile models...')
q10=train_q(0.10); print(f'  α=0.10 best={q10.best_iteration_}')
q50=train_q(0.50); print(f'  α=0.50 best={q50.best_iteration_}')
q90=train_q(0.90); print(f'  α=0.90 best={q90.best_iteration_}')

lo_val=np.expm1(np.clip(q10.predict(X_val_g),0,None))
hi_val=np.expm1(q90.predict(X_val_g))
lo_te =np.expm1(np.clip(q10.predict(X_test_g),0,None))
hi_te =np.expm1(q90.predict(X_test_g))

In [ ]:
# ── Conformal calibration ──────────────────────────────────────────────────────
def picp(y,lo,hi): return float(((y>=lo)&(y<=hi)).mean())
scores=np.maximum(lo_val-y_val, y_val-hi_val)
q_hat=float(np.quantile(scores, min(1.0, 0.80*(len(y_val)+1)/len(y_val))))
lo_cal=np.clip(lo_te-q_hat,0,None); hi_cal=hi_te+q_hat
print(f'Conformal q_hat : LKR {q_hat:,.0f}')
print(f'Raw PICP  test  : {picp(y_test,lo_te,hi_te)*100:.1f}%')
print(f'Calibrated PICP : {picp(y_test,lo_cal,hi_cal)*100:.1f}%  (target 80%)')

In [ ]:
# ── SHAP ──────────────────────────────────────────────────────────────────────
print(f'Computing SHAP on {PREDICTOR}...')
n_shap=min(3000,len(X_test_g))
idx=np.random.RandomState(42).choice(len(X_test_g),n_shap,replace=False)

if PREDICTOR=='CatBoost':
    explainer=shap.TreeExplainer(results['CatBoost']['model'])
    sf=results['CatBoost']['model'].get_feature_importance(
        Pool(X_test_cb.iloc[idx],label=y_val_log[:n_shap],cat_features=categorical_cols),type='ShapValues')
    sv=sf[:,:-1]; base_log=float(sf[0,-1]); X_shap=X_test_cb.iloc[idx]
else:
    explainer=shap.TreeExplainer(results[PREDICTOR]['model'])
    X_shap=X_test_g.iloc[idx]; sv=explainer.shap_values(X_shap)
    base_log=float(np.ravel(explainer.expected_value)[0])

mean_abs=np.abs(sv).mean(axis=0); top_ix=np.argsort(mean_abs)[::-1][:15]
print(f'Base log: {base_log:.4f}  → LKR {np.expm1(base_log):,.0f}')
print('\nTop 15 features (mean |SHAP|):')
for r,j in enumerate(top_ix,1):
    print(f'  {r:2d}. {X_shap.columns[j]:38s}  {mean_abs[j]:.4f}')

plt.figure(figsize=(10,8))
shap.summary_plot(sv,X_shap,max_display=15,show=False)
plt.title(f'SHAP — {PREDICTOR} Breakdown Cost v4 (top 15)',fontweight='bold')
plt.tight_layout(); plt.savefig('shap_summary_v4.png',dpi=120,bbox_inches='tight'); plt.close()
print('Saved: shap_summary_v4.png')

In [ ]:
# ── Success criteria ───────────────────────────────────────────────────────────
final_r2=test_metrics[PREDICTOR]['R²']
final_mape=test_metrics[PREDICTOR]['MAPE%']
final_picp=picp(y_test,lo_cal,hi_cal)
print('='*60+f'\nSUCCESS CRITERIA — Breakdown Cost v4 ({PREDICTOR})\n'+'='*60)
checks=[('R² ≥ 0.75',final_r2>=0.75,f'{final_r2:.4f}'),
        ('PICP@80% ∈ [78,82]',0.78<=final_picp<=0.82,f'{final_picp*100:.1f}%'),
        ('MAPE < 25%',final_mape<25,f'{final_mape:.2f}%')]
all_pass=all(p for _,p,_ in checks)
for name,p,v in checks: print(f'  {"✓ PASS" if p else "✗ FAIL"} | {name:25s} | {v}')
print('─'*60); print(f'  {"✓ ACCEPTED" if all_pass else "⚠ REVIEW"}'); print('='*60)

In [ ]:
# ── Save bundle ────────────────────────────────────────────────────────────────
FLEET_MEAN_LKR_V4 = float(df[TARGET].mean())
EXPECTED_RANGES_V4 = {}
for stype in df['next_service_type'].unique():
    sub = df[df['next_service_type']==stype][TARGET]
    if len(sub) > 20:
        EXPECTED_RANGES_V4[str(stype)] = (float(sub.quantile(0.25)), float(sub.quantile(0.75)))

bundle = {
    'version'            : MODEL_VERSION,
    'predictor_name'     : PREDICTOR,
    'target'             : TARGET,
    'target_description' : 'Predicted cost if asset requires maintenance in next 30 days',
    'feature_cols'       : feature_cols,
    'numeric_cols'       : numeric_cols,
    'categorical_cols'   : categorical_cols,
    'predictor_model'    : results[PREDICTOR]['model'],
    'q10': q10, 'q50': q50, 'q90': q90,
    'gbm_dtypes_safe'    : gbm_dtypes_safe,
    'shap_explainer'     : explainer,
    'conformal_q_hat'    : float(q_hat),
    'target_coverage'    : 0.80,
    'log1p_target'       : True,
    'test_metrics'       : test_metrics[PREDICTOR],
    'picp_80_test'       : float(final_picp),
    'fleet_mean_lkr'     : FLEET_MEAN_LKR_V4,
    'expected_ranges'    : EXPECTED_RANGES_V4,
    'train_rows'         : len(train_df),
    'val_rows'           : len(val_df),
    'test_rows'          : len(test_df),
    'dataset'            : 'v11 (filtered: maintenance_required=1, cost>0)',
    'vehicle_size_map'   : {'Mini_Truck_1T':1,'Delivery_Van_1.5T':2,'Light_Truck_3.5T':3,
                            'Medium_Truck_7T':4,'Heavy_Truck_16T':5,'Forklift_2.5T':3,'Forklift_3.0T':4},
    'part_tokens'        : ['oil_filter','engine_oil','brake_pads','discs','fluid','tires','valves',
                            'hydraulic_pump','seals','battery','terminals','radiator','coolant',
                            'hoses','injectors','gaskets','engine','clutch','mounts'],
}
joblib.dump(bundle,'predictix_breakdown_cost_model_v4.pkl')
print(f'Saved: predictix_breakdown_cost_model_v4.pkl')

decision_log={
    'version':MODEL_VERSION,'predictor':PREDICTOR,
    'target':TARGET,
    'dataset':'v11 filtered (maintenance_required=1)',
    'train_rows':len(train_df),'val_rows':len(val_df),'test_rows':len(test_df),
    'test_metrics':{k:{m:round(float(v),4) for m,v in test_metrics[k].items()} for k in test_metrics},
    'picp_calibrated':round(float(final_picp),4),
    'conformal_q_hat':round(float(q_hat),2),
    'fleet_mean_lkr':round(FLEET_MEAN_LKR_V4,2),
    'success':{'all_pass':bool(all_pass)},
    'top15_shap':[{'rank':r+1,'feature':X_shap.columns[j],'mean_abs_shap':round(float(mean_abs[j]),4)}
                   for r,j in enumerate(top_ix)],
}
import json
with open('predictix_breakdown_cost_model_v4_log.json','w') as f:
    json.dump(decision_log,f,indent=2,default=str)
print('Saved: predictix_breakdown_cost_model_v4_log.json')

In [ ]:
# ── Inference demo on SLW0668 profile ─────────────────────────────────────────
b = joblib.load('predictix_breakdown_cost_model_v4.pkl')
gbm_dtypes = {c: pd.CategoricalDtype(cats, ordered=False) for c,cats in b['gbm_dtypes_safe'].items()}

def predict_breakdown_cost(raw_input, bundle, top_k=5):
    import pandas as pd, numpy as np
    from catboost import Pool

    d = dict(raw_input)
    age = float(d.get('vehicle_age_years',0))
    vt  = str(d.get('vehicle_type',''))
    nsv = str(d.get('next_service_type', d.get('last_service_type','oil_service')))
    VSMAP = bundle['vehicle_size_map']

    d['vehicle_size_ord']   = VSMAP.get(vt,3)
    d['age_band']           = 'new' if age<=3 else 'mid' if age<=6 else 'old' if age<=10 else 'aged'
    d['service_vehicle']    = f'{nsv}__{vt}'
    d['payload_age']        = float(d.get('payload_capacity_kg',0))*age
    d['utilisation_stress'] = float(d.get('payload_utilization_pct',0))*float(d.get('downtime_hours_last_90d',0))
    d['health_index']       = (float(d.get('oil_life_pct',50))+float(d.get('brake_health_pct',50))+
                               float(d.get('tire_health_pct',50))+float(d.get('battery_health_pct',50)))/4
    d['fault_stress']       = float(d.get('active_fault_code_count',0))+float(d.get('sensor_fault_flag',0))*3
    d['health_deficit']     = 100 - d['health_index']
    d['overdue_days']       = max(0, float(d.get('days_since_last_service',0))-60)
    d['breakdown_history']  = float(d.get('lifetime_breakdown_count',0))
    d['service_intensity']  = float(d.get('lifetime_service_count',0))/(age+1)
    for tok in bundle['part_tokens']:
        d[f'part_{tok}'] = int(tok in str(d.get('parts_replaced_last_service','')))

    row={c:d.get(c) for c in bundle['feature_cols']}
    X=pd.DataFrame([row])
    Xc=X.copy()
    for c in bundle['categorical_cols']: Xc[c]=Xc[c].astype(str).fillna('__missing__')
    for c in bundle['numeric_cols']:     Xc[c]=pd.to_numeric(Xc[c],errors='coerce').fillna(0.0)
    Xc=Xc[bundle['feature_cols']]
    gdtypes={c:pd.CategoricalDtype(cats,ordered=False) for c,cats in bundle['gbm_dtypes_safe'].items()}
    Xg=X.copy()
    for c in bundle['categorical_cols']: Xg[c]=Xg[c].astype(str).fillna('__missing__').astype(gdtypes[c])
    for c in bundle['numeric_cols']:     Xg[c]=pd.to_numeric(Xg[c],errors='coerce').fillna(0.0)
    Xg=Xg[bundle['feature_cols']]

    log_pred=float(bundle['predictor_model'].predict(Xc)[0])
    point=float(np.expm1(max(0,log_pred)))
    lo=float(max(0,np.expm1(max(0,float(bundle['q10'].predict(Xg)[0])))-bundle['conformal_q_hat']))
    hi=float(np.expm1(float(bundle['q90'].predict(Xg)[0]))+bundle['conformal_q_hat'])

    if bundle['predictor_name']=='CatBoost':
        sf=bundle['predictor_model'].get_feature_importance(
            Pool(Xc,cat_features=bundle['categorical_cols']),type='ShapValues')
        sv_arr=sf[0,:-1]
    else:
        sv_arr=bundle['shap_explainer'].shap_values(Xg)[0]
    total_abs=float(np.abs(sv_arr).sum()) or 1e-9
    top=np.argsort(np.abs(sv_arr))[::-1][:top_k]
    drivers=[{'feature':Xc.columns[j],'value':str(Xc.iloc[0,j]),
              'direction':'increases' if sv_arr[j]>0 else 'decreases',
              'relative_impact':round(abs(float(sv_arr[j]))/total_abs*100,1),
              'sv_log':round(float(sv_arr[j]),6)} for j in top]

    rng=bundle['expected_ranges'].get(nsv)
    sanity=('within_expected' if rng and rng[0]<=point<=rng[1]
            else 'below_expected' if rng and point<rng[0] else 'above_expected' if rng else 'unknown')
    return {'predicted_cost_lkr':round(point,2),'pi_80_lower_lkr':round(lo,2),
            'pi_80_upper_lkr':round(hi,2),'coverage_target':'80%',
            'fleet_mean_lkr':round(bundle['fleet_mean_lkr'],2),
            'vs_fleet_mean_lkr':round(point-bundle['fleet_mean_lkr'],2),
            'expected_range':{'p25_lkr':round(rng[0],0),'p75_lkr':round(rng[1],0)} if rng else {},
            'sanity_check':sanity,'top_drivers':drivers}

# SLW0668 profile
slw0668 = {
    'vehicle_type':'Forklift_2.5T','next_service_type':'brake_service',
    'vehicle_role':'internal_material_handling','make_model':'Heli CPCD25','fuel_type':'Diesel',
    'transmission':'Manual','manufacture_year':2014,'vehicle_age_years':11.0,
    'payload_capacity_kg':2502.40,'odometer_km':19128,'engine_hours_total':4500,
    'oil_life_pct':77,'brake_health_pct':74,'tire_health_pct':50,'battery_health_pct':77,
    'hydraulic_health_pct':73,'active_fault_code_count':1,'sensor_fault_flag':0,
    'lifetime_service_count':40,'lifetime_breakdown_count':3,
    'days_since_last_service':102,'engine_hours_since_last_service':300,
    'mileage_since_last_service_km':1200,'downtime_hours_last_90d':4.2,
    'payload_utilization_pct':70,'fuel_price_lkr_per_l':310.0,
    'service_provider_type':'third_party','maintenance_priority':'Medium',
    'route_type':'Urban','cargo_type':'Industrial_goods','operating_shift':'Day',
    'parts_replaced_last_service':'brake_pads;fluid',
    'avg_payload_kg':1750,'overload_events_30d':0,'distance_last_30d_km':800,
    'operating_hours_last_30d':110,'idle_hours_last_30d':18,'trip_count_30d':40,
    'avg_trip_distance_km':20,'start_stop_burden_30d':80,'rough_road_pct':10,
    'urban_route_pct':80,'port_route_pct':5,'fuel_rate_lph':5.2,
    'fuel_efficiency_km_per_l':4.1,'engine_temp_avg_c':85,'coolant_temp_max_c':92,
    'vibration_rms_mm_s':2.1,'tire_pressure_psi':95,'battery_voltage_v':12.4,
    'ambient_temp_avg_c':30,'ambient_humidity_avg_pct':75,'rainfall_mm_30d':100,
}

r = predict_breakdown_cost(slw0668, b, top_k=5)
print('\n=== SLW0668 Breakdown Cost Prediction ===')
print(f'  Predicted cost : LKR {r["predicted_cost_lkr"]:>12,.2f}')
print(f'  80% PI         : LKR {r["pi_80_lower_lkr"]:>10,.2f}  →  LKR {r["pi_80_upper_lkr"]:,.2f}')
print(f'  Fleet mean     : LKR {r["fleet_mean_lkr"]:>12,.2f}')
print(f'  vs fleet mean  : LKR {r["vs_fleet_mean_lkr"]:>+12,.2f}')
print(f'  Sanity check   : {r["sanity_check"]}')
print(f'\n  Top SHAP drivers:')
for d in r['top_drivers']:
    icon='▲' if d['direction']=='increases' else '▼'
    print(f'  {icon} {d["feature"]:<34s}  {d["relative_impact"]:>5.1f}%  val={d["value"]}')


## Summary

| Property | Value |
|---|---|
| **Version** | `breakdown-cost-v4.0` |
| **Target** | `maintenance_cost_lkr_next_30d` — forward-looking breakdown cost |
| **Training rows** | Only rows where `maintenance_required_next_30d == 1` and cost > 0 |
| **Aligns with SHAP** | Yes — uses same health signals (oil life, tire health, battery voltage, days overdue) as the failure model |
| **New engineered features** | `health_deficit`, `overdue_days`, `breakdown_history`, `service_intensity` |
| **PI** | LightGBM quantile α=0.10/0.50/0.90 + conformal calibration |
| **SHAP** | `relative_impact %` — always bounded, cross-asset consistent |
